In [1]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
for candidate in [
    root,
    root / "apps" / "api",
    root.parent,
    root.parent / "apps" / "api",
]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.append(str(candidate))

In [7]:
from core.db import engine
from sqlalchemy import text

# Teste

In [2]:
import httpx

In [3]:
BASE_URL = 'http://localhost:8000'

In [9]:
response = httpx.get(f'{BASE_URL}/works/?offset=0&limit=20')
print(response.status_code)
print(response.json())

200
[{'title': 'As pequenas virtudes', 'type': 'Livro', 'subject': 'Literatura', 'id': '825ebba7-93cc-4768-bc2a-30fd4c4ca384'}]


In [4]:
payload = {
    "title": "As pequenas virtudes",
    "type": "Livro",
    "subject": "Literatura"
}

response = httpx.post("http://localhost:8000/works/", json=payload)
print(response.status_code)
print(response.json())

201
{'title': 'As pequenas virtudes', 'type': 'Livro', 'subject': 'Literatura', 'id': '825ebba7-93cc-4768-bc2a-30fd4c4ca384'}


In [5]:
payload = {
  "name": "Natália Ginzburg",
  "type": "Person",
  "identifier": "A5059313957"
}

response = httpx.post("http://localhost:8000/agents/", json=payload)
print(response.status_code)
print(response.json())

201
{'id': '5d0d229f-3f25-4455-8e22-01960cc1db44', 'name': 'Natália Ginzburg', 'type': 'Person', 'identifier': 'A5059313957'}


In [6]:
work_id = '825ebba7-93cc-4768-bc2a-30fd4c4ca384'
payload = {
  "agent_id": "5d0d229f-3f25-4455-8e22-01960cc1db44",
  "role": "Autor"
}

response = httpx.post(
    f"{BASE_URL}/works/{work_id}/agents", 
    json=payload)
print(response.status_code)
print(response.json())

201
{'agent_id': '5d0d229f-3f25-4455-8e22-01960cc1db44', 'role': 'Autor', 'work_id': '825ebba7-93cc-4768-bc2a-30fd4c4ca384'}


In [ ]:
payload = {
  "work_id": work_id,
  "isbn": "978-8535932973",
  "publication_year": 2026,
  "formato": "impresso"
}

response = httpx.post(
    f"{BASE_URL}/instances", 
    json=payload)
print(response.status_code)
print(response.json())

307


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [8]:
response

<Response [307 Temporary Redirect]>

In [16]:
data = WorkAgentCreate(**payload)
link = WorkAgent(work_id=work_id, agent_id=data.agent_id, role=data.role)

In [2]:
from core.db import get_db
from sqlalchemy.ext.asyncio import AsyncSession


db: AsyncSession = get_db()


In [5]:
from sqlalchemy.ext.asyncio import (
    AsyncSession,
    async_sessionmaker,
    create_async_engine,
)

In [10]:
SessionLocal = async_sessionmaker(
    bind=engine,
    class_=AsyncSession,
    expire_on_commit=False,
)

async def get_db():
    async with SessionLocal() as session:
        yield session
        
        
db = get_db()

In [11]:
from services.crud.work import list_works


l = await list_works(db)

AttributeError: 'async_generator' object has no attribute 'execute'

In [16]:
from sqlalchemy import select

from models.work import Work

async with engine.begin() as conn:
        result = await conn.execute(select(Work))
        print(result)

2026-07-22 19:05:05,387 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-22 19:05:05,412 INFO sqlalchemy.engine.Engine ROLLBACK


InvalidRequestError: Mapper 'Mapper[Agent(agent)]' has no property 'published_instances'.  If this property was indicated from other mappers or configure events, ensure registry.configure() has been called.